# មេរៀន ១១ - អ្នកតំណាងទៅអ្នកតំណាង (A2A) ពិធីការ


## ការតំឡើង


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## តើកម្មវិធីសមាសភាព A2A ជាអ្វី?

កម្មវិធីសមាសភាព **Agent-to-Agent (A2A)** គឺជាមาตรฐานចំហ ដែលអនុញ្ញាតឱ្យភ្នាក់ងារ AI ទំនាក់ទំនង,
ស្វែងរកគ្នា និងសហការគ្នា — ទោះបីជាពួកគេត្រូវបានបង្កើតលើស្នាដៃខុសគ្នា ឬផ្ទុក
ដោយសេវាកម្មខុសគ្នាក៏ដោយ។

គំនិតសំខាន់ៗ៖

- **ការស្វែងរក** – ភ្នាក់ងារបោះពុម្ភ *កាតភ្នាក់ងារ* ដែលពិពណ៌នាពីសមត្ថភាពរបស់ពួកគេ ធ្វើឱ្យ
  ងាយស្រួលសម្រាប់ភ្នាក់ងារផ្សេងទៀត (ឬអ្នករៀបចំការ) ដើម្បីស្វែងរកអ្នកជំនាញសមរម្យសម្រាប់ភារកិច្ចមួយ។
- **ការផ្ទេរសារអក្សរ** – ភ្នាក់ងារប្ដូរសារដែលមានរចនាសម្ព័ន្ធតាមរយៈកម្មវិធីសមាសភាពទាំងមូល ដូច្នេះ
  សំណើពីភ្នាក់ងារមួយអាចត្រូវបានយល់ និងបំពេញដោយភ្នាក់ងារផ្សេងទៀតដោយមិនគិតពីអនុវត្តក្នុងផ្ទៃ។

- **ជីវិតវដ្តភារកិច្ច** – A2A កំណត់ស្ថានភាពដូចជា *បានដាក់ស្នើ*, *កំពុងធ្វើការ*, *បានបញ្ចប់*, និង
  *បរាជ័យ*, ដើម្បីឲ្យអ្នករៀបចំមានទិដ្ឋភាពពេញលេញអំពីការរីកចម្រើននៃភារកិច្ចដែលបានចែកចាយ។

នៅក្នុងមេរៀននេះ យើងធ្វើឱ្យការសហការរបស់បែប A2A ដោយភ្ជាប់ភ្នាក់ងារជំនាញដំណើរកំសាន្តបីរូប
ទៅក្នុងលំនាំការងារដែលភ្នាក់ងារ​កុំព្រងចំនេះដឹកនាំជំនាញរបស់ខ្លួន ហើយផ្ទេរវិលតំណរ​ទៅភ្នាក់ងារចាប់ផ្តើមបន្ទាប់។


## ការបង្កើតភ្នាក់ងារធ្វើដំណើរដែលមានឯកទេស


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## ការសហការរវាងភ្នាក់ងារច្រើនតាមរយៈនីតិវិធី

យើងភ្ជាប់ភ្នាក់ងារទាំងបីទៅជានីតិវិធីតម្រៀបទៅមុខដែលស្រដៀងនឹងការផ្ទេរសារ A2A៖

1. **CurrencyExchangeAgent** ទទួលសំណើរពីអ្នកប្រើនិងបង្កើតទ្រង់ទ្រាយប្រាក់បន្លាស់។
2. **ActivityPlannerAgent** ទទួលព្រឹត្តិប័ត្រដែលបានបន្ថែមនិងបន្ថែមការណែនាំសកម្មភាព។
3. **TravelManagerAgent** បញ្ចូលការបញ្ចូលទាំងពីរជារបាយការណ៍ដំណើរកម្សាន្តចុងក្រោយ។


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## ការយល់ដឹងអំពី A2A ក្នុងការផលិត

នៅក្នុងបរិយាកាសផលិតកម្ម ពហុបច្ចេកវិទ្យា A2A បើកដំណើរការករណីប្រើប្រាស់ឆ្លងសេវាកម្មដ៏មានអំណាច៖

| សមត្ថភាព | សេចក្តីពិពណ៌នា |
|---|---|
| **ការចល័តឆ្លងស៊ុមបច្ចេកវិទ្យា** | អ្នកតំណាងដែលបានបង្កើតជាមួយពហុបច្ចេកវិទ្យាមួយអាចផ្គល់ភារកិច្ចទៅអ្នកតំណាងដែលបានបង្កើតជាមួយបច្ចេកវិទ្យាផ្សេងទៀតដែលគោរពទៅ A2A បាន ដើម្បីអនុញ្ញាតឱ្យមានការសហប្រតិបត្តិការពិតប្រាកដរវាងអង្គភាពផ្សេងគ្នា។ |
| **ព្រំប្រទល់សេវាកម្ម** | អ្នកតំណាងអាចរស់នៅក្នុងមីក្រូសេវាកម្មផ្សេងៗ តំបន់ពពកផ្សេងៗ មិនថាជាអង្គភាពផ្សេងៗ ឬក៏ស្ថិតនៅក្នុងអង្គភាពផ្សេងៗយ៉ាងដូចម្តេចក៏ដោយ ខណៈពេលដែលនៅតែសហការបានយ៉ាងរលូន។ |
| **ការស្វែងរកឆ្លាតវៃ** | អ្នករៀបចំអាចសាកសួរតារាងកាតអ្នកតំណាង (Agent Card registry) នៅពេលដំណើរការដើម្បីស្វែងរកអ្នកឯកទេសសម្រាប់ភារកិច្ចតូចៗមួយ។ |
| **ការផ្សាយបន្ត និងការជូនដំណឹងរុញចុះ** | A2A គាំទ្រកម្មវិធីប្រាប់ដំណឹងពីម៉ាស៊ីនបម្រើ (Server-Sent Events - SSE) សម្រាប់បញ្ជូនព័ត៌មានលំអិតក្នុងពេលវេលាពិត និងការជូនដំណឹងរុញសម្រាប់ភារកិច្ចរយៈពេលវែង។ |

ដំណើរការដែលយើងបានបង្កើតខាងលើគឺជារូបមន្តប៉ុណ្ណោះនៃទ្រង់ទ្រាយនេះ។ នៅក្នុងការតំឡើងពិតប្រាកដ
អ្នកតំណាងនីមួយៗនឹងបង្ហាញចំណុចភ្ជាប់ HTTP មួយ បោះពុម្ព Agent Card និងទំនាក់ទំនង
តាមរយៈពហុបច្ចេកវិទ្យា JSON-RPC របស់ A2A។


## សង្ខេប

នៅក្នុងមេរៀននេះ អ្នកបានរៀន៖

1. **ត Protocol A2A ជាអ្វី** — ស្តង់ដារបើកសម្រាប់ការរកឃើញភ្នាក់ងារទៅភ្នាក់ងារមួយ, ការ​ផ្ញើសារនិង
   ការគ្រប់គ្រងភារកិច្ច។
2. **របៀបបង្កើតភ្នាក់ងារដែលមានឯកទេស** — ភ្នាក់ងារប្ដូរប្រាក់បរទេស, ភ្នាក់ងាររៀបចំសកម្មភាព,
   និងអ្នកគ្រប់គ្រងការធ្វើដំណើរតាមប្រព័ន្ធ។
3. **របៀបភ្ជាប់ភ្នាក់ងារចូលទៅក្នុងសកម្មភាពមួយ** — ការប្រើ `WorkflowBuilder` ដើម្បីគំរូ
   ការផ្ញើសារតាមលំដាប់រវាងភ្នាក់ងារ។
4. **របៀបដំណើរការ A2A ក្នុងផលិតកម្ម** — អនុញ្ញាតឲ្យសហការលើសពីប្លាទផមកុំព្យូទ័រនិងសេវាកម្មផ្សេងៗ
   ជាមួយការរកឃើញឌីណាមាញនិងការផ្សាយព័ត៌មានបន្តជាបន្តបន្ទាប់។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
